In [7]:
import torch


def detect_objects(model, image_tensor, score_threshold=0.5):
    """
    Run object detection on an image
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with torch.no_grad():
        predictions = model([image_tensor.to(device)])
    
    boxes = predictions[0]['boxes'].cpu().numpy()
    scores = predictions[0]['scores'].cpu().numpy()
    labels = predictions[0]['labels'].cpu().numpy()
    
    # Filter by confidence threshold
    mask = scores > score_threshold
    boxes = boxes[mask]
    scores = scores[mask]
    labels = labels[mask]
    
    return boxes, scores, labels

# Example with sample image
from PIL import Image
import torchvision.transforms as transforms

# Load and preprocess a sample image
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Create dummy image for demonstration
dummy_image = torch.rand(3, 480, 640)

# Run detection
boxes, scores, labels = detect_objects(model, dummy_image, score_threshold=0.5)

print(f"Detected {len(boxes)} objects")
print(f"Boxes shape: {boxes.shape}")
print(f"Scores: {scores[:5] if len(scores) > 0 else 'None'}")

Detected 0 objects
Boxes shape: (0, 4)
Scores: None


## 3. Detection on Sample Images

In [6]:
# YOLOv5 example (requires: pip install yolov5)
# Uncomment to use actual YOLOv5
# import yolov5
# model = yolov5.load('yolov5s.pt')

# Faster R-CNN from torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load pre-trained Faster R-CNN
model = fasterrcnn_resnet50_fpn(pretrained=True)
model = model.to(device)
model.eval()

print("Faster R-CNN loaded successfully")
print(f"Model: {model}")

c:\Users\adity\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\adity\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\adity/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:01<00:00, 114MB/s]  


Faster R-CNN loaded successfully
Model: FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm

## 2. Using Pre-trained YOLOv5

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# IoU (Intersection over Union) calculation
def compute_iou(box1, box2):
    """
    Compute IoU between two boxes
    box format: [x1, y1, x2, y2]
    """
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    
    # Intersection area
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    inter_area = max(0, inter_x_max - inter_x_min) * max(0, inter_y_max - inter_y_min)
    
    # Union area
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    union_area = box1_area + box2_area - inter_area
    
    iou = inter_area / union_area if union_area > 0 else 0
    return iou

# Non-Maximum Suppression (NMS)
def nms(boxes, scores, iou_threshold=0.5):
    """
    Apply NMS to remove overlapping boxes
    """
    if len(boxes) == 0:
        return []
    
    sorted_indices = np.argsort(scores)[::-1]
    keep = []
    
    while len(sorted_indices) > 0:
        current = sorted_indices[0]
        keep.append(current)
        
        if len(sorted_indices) == 1:
            break
        
        current_box = boxes[current]
        remaining_boxes = boxes[sorted_indices[1:]]
        ious = np.array([compute_iou(current_box, box) for box in remaining_boxes])
        
        sorted_indices = sorted_indices[1:][ious < iou_threshold]
    
    return keep

print("IoU and NMS functions defined successfully")

IoU and NMS functions defined successfully


## 1. Object Detection Fundamentals

# Object Detection
## Detecting and Localizing Multiple Objects in Images

Topics covered:
- Bounding box regression and IoU calculations
- Anchor-based detection (YOLO, Faster R-CNN)
- Anchor-free detection (FCOS)
- Non-Maximum Suppression (NMS)
- Focal loss for class imbalance
- Training and evaluation on COCO/Pascal VOC
- Real-time detection pipelines